## Simulation Without PCA — Real-Time IDS Pipeline

### Flow:
1. Load preprocessed data (`modified_daata.csv`)
2. Remove irrelevant columns (Flow ID, Source IP, Destination IP, Timestamp, Label)
3. Sample `n` rows to simulate incoming traffic
4. Load StandardScaler artifact and scale data
5. Select the feature columns used during model training
6. Load XGBoost model (no PCA) and predict
7. Decode encoded labels back to human-readable strings
8. Stream predictions to Power BI Streaming Dataset

In [ ]:
import pandas as pd
import numpy as np
import joblib
import requests
import json
import time
from xgboost import XGBClassifier

In [ ]:
DATA_PATH = r'C:/Users/HP/OneDrive/Desktop/Data-Analysis-for-Network-Traffic-Analysis-main/Data/modified_daata.csv'
MODEL_DIR = r'C:/Users/HP/OneDrive/Desktop/Data-Analysis-for-Network-Traffic-Analysis-main/Packages and Models'

df = pd.read_csv(DATA_PATH)
# Strip all column name whitespace upfront — prevents KeyError on spaced column names
df.columns = df.columns.str.strip()
print(f"Loaded data: {df.shape}")
df.head()

In [ ]:
# Drop irrelevant columns (IDs, IPs, Timestamps, Label — these are not features)
cols_to_drop = ['Flow ID', 'Timestamp', 'Source IP', 'Destination IP', 'Label']
df = df.drop(columns=cols_to_drop, errors='ignore')
print(f"After dropping irrelevant cols: {df.shape}")
print("Feature columns:", df.columns.tolist())

### Step: Sample Traffic — Simulating Incoming Network Flows

In [ ]:
n = 100  # Number of flows to simulate — change as needed
simulated_data = df.sample(n, random_state=None).reset_index(drop=True)
print(f"Simulated {n} traffic flows.")
simulated_data.head()

### Step: Load StandardScaler and Scale Features

In [ ]:
scaler = joblib.load(f'{MODEL_DIR}/stdscaler.pkl')

# The scaler was trained on specific columns; match them
try:
    expected_features = list(scaler.feature_names_in_)
    # Reorder to match scaler training order
    simulated_data = simulated_data[expected_features]
    print(f"Reordered to match scaler's {len(expected_features)} training features.")
except AttributeError:
    print("ℹ️ Scaler has no feature_names_in_ (older sklearn). Using current column order.")

scaled_data = scaler.transform(simulated_data)
df_scaled = pd.DataFrame(scaled_data, columns=simulated_data.columns)
print(f"Scaled shape: {df_scaled.shape}")
df_scaled.head()

### Step: Load XGBoost Model (No PCA) and Predict

In [ ]:
xgb = XGBClassifier()
xgb.load_model(f'{MODEL_DIR}/xgb_model.model')

predict_encoded = xgb.predict(df_scaled.values)
print(f"Predictions (encoded): {predict_encoded[:10]}")

### Step: Decode Predictions to Human-Readable Labels

In [ ]:
lb = joblib.load(f'{MODEL_DIR}/labelencoded.pkl')
predict_labels = lb.inverse_transform(predict_encoded)
print("Sample predictions:", predict_labels[:10])

print(f"\nAttack distribution in simulation:")
print(pd.Series(predict_labels).value_counts())

### Step: Select Payload Columns for Power BI
We send a subset of meaningful feature columns alongside the prediction to Power BI for rich dashboard visuals.

⚠️ **Replace `POWERBI_URL`** with your own Power BI Streaming Dataset push URL before running.

In [ ]:
# Columns to include in the Power BI payload (for dashboard richness)
PAYLOAD_COLUMNS = [
    "Source Port", "Destination Port", "Protocol", "Flow Duration",
    "Total Fwd Packets", "Total Backward Packets",
    "Total Length of Fwd Packets", "Total Length of Bwd Packets",
    "Fwd Packet Length Max", "Fwd Packet Length Min",
    "Fwd Packet Length Mean", "Fwd Packet Length Std",
    "Bwd Packet Length Max", "Bwd Packet Length Min",
    "Bwd Packet Length Mean", "Bwd Packet Length Std",
    "Flow Packets/s",
    "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max", "Flow IAT Min",
    "Fwd IAT Total", "Fwd IAT Mean", "Fwd IAT Std", "Fwd IAT Max", "Fwd IAT Min",
    "Bwd IAT Total", "Bwd IAT Mean", "Bwd IAT Std", "Bwd IAT Max", "Bwd IAT Min",
    "Min Packet Length", "Max Packet Length", "Packet Length Mean", "Packet Length Std",
    "FIN Flag Count", "SYN Flag Count", "RST Flag Count",
    "PSH Flag Count", "ACK Flag Count", "URG Flag Count",
    "CWE Flag Count", "ECE Flag Count",
    "Down/Up Ratio", "Average Packet Size",
    "Active Mean", "Active Std", "Idle Mean", "Idle Std",
    "Flow Bytes/s"
]

# Only keep columns that exist in df_scaled
available_cols = [c for c in PAYLOAD_COLUMNS if c in df_scaled.columns]
missing_cols = [c for c in PAYLOAD_COLUMNS if c not in df_scaled.columns]
if missing_cols:
    print(f"⚠️ Missing payload columns (skipped): {missing_cols}")

payload_df = df_scaled[available_cols].copy()
payload_df['Predict_Label'] = predict_labels
print(f"Payload shape: {payload_df.shape}")
payload_df.head()

In [ ]:
# ─── CONFIGURE YOUR POWER BI URL BELOW ───────────────────────────────────────
POWERBI_URL = "https://api.powerbi.com/beta/YOUR_TENANT_ID/datasets/YOUR_DATASET_ID/rows?experience=power-bi&key=YOUR_API_KEY"
# ─────────────────────────────────────────────────────────────────────────────

headers = {"Content-Type": "application/json"}

success_count = 0
fail_count = 0

for idx in range(len(payload_df)):
    row = payload_df.iloc[idx]
    
    # Build payload — convert numpy types to native Python for JSON serialization
    payload = {}
    for col in available_cols:
        val = row[col]
        payload[col] = float(val) if not isinstance(val, str) else val
    payload["Predict_Label"] = str(predict_labels[idx])
    payload["Row_Index"] = int(idx)
    
    try:
        response = requests.post(POWERBI_URL, headers=headers, data=json.dumps([payload]), timeout=10)
        if response.status_code in [200, 202]:
            success_count += 1
            print(f"✅ Row {idx:3d} → {predict_labels[idx]}")
        else:
            fail_count += 1
            print(f"❌ Row {idx:3d} failed: {response.status_code} — {response.text[:80]}")
    except requests.exceptions.RequestException as e:
        fail_count += 1
        print(f"⚠️ Row {idx:3d} exception: {e}")
    
    time.sleep(0.05)  # Small delay to avoid rate-limiting

print(f"\n📊 Streaming complete: {success_count} sent, {fail_count} failed out of {len(payload_df)} rows.")

### Summary
- Loaded `modified_daata.csv` and sampled `n` traffic flows
- Applied saved StandardScaler (no PCA — full feature space)
- Predicted using XGBoost (full features) model
- Decoded predictions using LabelEncoder
- Streamed each flow + label to Power BI Streaming Dataset

**Use this notebook for maximum accuracy (no dimension reduction).**
**Use `Simulation_PCA.ipynb` for faster inference with minimal accuracy trade-off.**